# SOC Triage Dataset Creation and Evaluation

## 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

DATA_DIR = Path("/content")
OUTPUT_DIR = DATA_DIR / "Evaluation_Outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("Data folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)


## 2. Load Uploaded Source Files

In [ ]:

SCENARIO_FILE = DATA_DIR / "SOC_Scenario_Register_Updated.xlsx"
MANUAL_FILE = DATA_DIR / "Manual_Triage_Results.xlsx"
RULE_FILE = DATA_DIR / "n8n_Rule_Based_Results.xlsx"
GEMINI_FILE = DATA_DIR / "Gemini_LLM_Assisted_Results.xlsx"
REFERENCE_FILE = DATA_DIR / "Reference_Outcomes.xlsx"

source_files = [
    SCENARIO_FILE,
    MANUAL_FILE,
    RULE_FILE,
    GEMINI_FILE,
    REFERENCE_FILE
]

for file in source_files:
    if not file.exists():
        raise FileNotFoundError(f"Missing file: {file}")

scenario = pd.read_excel(SCENARIO_FILE, sheet_name="Scenario Register")
manual = pd.read_excel(MANUAL_FILE, sheet_name="Manual Results")
rule = pd.read_excel(RULE_FILE, sheet_name="n8n Rule-Based Results")
gemini = pd.read_excel(GEMINI_FILE, sheet_name="LLM-Assisted Results")
reference = pd.read_excel(REFERENCE_FILE, sheet_name="Reference Outcomes")

for df in [scenario, manual, rule, gemini, reference]:
    df.columns = df.columns.astype(str).str.strip()

print("Scenario Register:", scenario.shape)
print("Manual Results:", manual.shape)
print("Rule-Based Results:", rule.shape)
print("Gemini Results:", gemini.shape)
print("Reference Outcomes:", reference.shape)


## 3. Create Clean Dataset

In [ ]:

def parse_seconds(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.number)):
        return float(value)

    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", str(value))
    return float(match.group(1)) if match else np.nan


def is_na(value):
    if pd.isna(value):
        return True
    return str(value).strip().upper() == "N/A"


scenario_info = scenario.set_index("Case ID")
manual_info = manual.set_index("Case ID")
rule_info = rule.set_index("Case ID")
gemini_info = gemini.set_index("Case ID")
reference_info = reference.set_index("Case ID")

rows = []

for i in range(1, 13):
    case_id = f"SC-{i:02d}"

    category = scenario_info.loc[case_id, "Category"]
    scenario_name = scenario_info.loc[case_id, "Scenario"]
    scenario_type = scenario_info.loc[case_id, "Type"]

    ref_decision = reference_info.loc[case_id, "Reference Decision"]
    ref_severity = reference_info.loc[case_id, "Reference Severity"]
    ref_escalation = reference_info.loc[case_id, "Reference Escalation"]

    manual_decision = manual_info.loc[case_id, "Decision"]
    rule_decision = rule_info.loc[case_id, "Decision"]
    gemini_decision = gemini_info.loc[case_id, "Decision"]

    not_executed = (
        is_na(manual_decision)
        and is_na(rule_decision)
        and is_na(gemini_decision)
    )

    status = "Not Executed" if not_executed else "Completed"
    include = "No" if not_executed else "Yes"

    rows.append({
        "Case ID": case_id,
        "Category": category,
        "Scenario": scenario_name,
        "Scenario Type": scenario_type,
        "Status": status,
        "Method": "Manual",
        "Decision": manual_decision,
        "Severity": manual_info.loc[case_id, "Severity"],
        "Escalation": manual_info.loc[case_id, "Escalation"],
        "Time (s)": parse_seconds(manual_info.loc[case_id, "Time (seconds)"]),
        "Reference Decision": ref_decision,
        "Reference Severity": ref_severity,
        "Reference Escalation": ref_escalation,
        "Include in Metrics": include,
        "Gemini Confidence (%)": np.nan
    })

    rows.append({
        "Case ID": case_id,
        "Category": category,
        "Scenario": scenario_name,
        "Scenario Type": scenario_type,
        "Status": status,
        "Method": "Rule-Based",
        "Decision": rule_decision,
        "Severity": rule_info.loc[case_id, "Severity"],
        "Escalation": rule_info.loc[case_id, "Escalation"],
        "Time (s)": parse_seconds(rule_info.loc[case_id, "Execution Time (s)"]),
        "Reference Decision": ref_decision,
        "Reference Severity": ref_severity,
        "Reference Escalation": ref_escalation,
        "Include in Metrics": include,
        "Gemini Confidence (%)": np.nan
    })

    rows.append({
        "Case ID": case_id,
        "Category": category,
        "Scenario": scenario_name,
        "Scenario Type": scenario_type,
        "Status": status,
        "Method": "Gemini",
        "Decision": gemini_decision,
        "Severity": gemini_info.loc[case_id, "Severity"],
        "Escalation": gemini_info.loc[case_id, "Escalation"],
        "Time (s)": parse_seconds(gemini_info.loc[case_id, "Execution time"]),
        "Reference Decision": ref_decision,
        "Reference Severity": ref_severity,
        "Reference Escalation": ref_escalation,
        "Include in Metrics": include,
        "Gemini Confidence (%)": gemini_info.loc[case_id, "Confidence"]
    })

dataset = pd.DataFrame(rows)

CLEAN_XLSX = DATA_DIR / "SOC_Triage_Clean_Dataset.xlsx"
CLEAN_CSV = DATA_DIR / "SOC_Triage_Clean_Dataset.csv"

dataset.to_excel(CLEAN_XLSX, index=False)
dataset.to_csv(CLEAN_CSV, index=False)

display(dataset)

print("Saved:", CLEAN_XLSX)
print("Saved:", CLEAN_CSV)


## 4. Prepare Evaluation Data

In [ ]:

analysis = dataset[dataset["Include in Metrics"] == "Yes"].copy()

DECISION_LABELS = ["Benign", "Investigate", "Escalate"]
SEVERITY_MAP = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
ESCALATION_MAP = {
    "No escalation": 0,
    "Investigate further": 1,
    "Escalate to L2": 2
}
METHODS = ["Manual", "Rule-Based", "Gemini"]

print("Executed scenarios:", analysis["Case ID"].nunique())
print("Evaluation rows:", len(analysis))

display(
    dataset[
        ["Case ID", "Status", "Include in Metrics"]
    ].drop_duplicates()
)


## 5. Decision Performance

In [ ]:

decision_rows = []

for method in METHODS:
    x = analysis[analysis["Method"] == method]
    y_true = x["Reference Decision"]
    y_pred = x["Decision"]

    decision_rows.append({
        "Method": method,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(
            y_true,
            y_pred,
            labels=DECISION_LABELS,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_true,
            y_pred,
            labels=DECISION_LABELS,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            labels=DECISION_LABELS,
            average="macro",
            zero_division=0
        )
    })

decision_metrics = pd.DataFrame(decision_rows)
display(decision_metrics.round(4))

ax = decision_metrics.set_index("Method").plot(
    kind="bar",
    y=["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"],
    figsize=(9, 5)
)

ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Decision Performance")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Decision_Performance.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

for method in METHODS:
    x = analysis[analysis["Method"] == method]

    cm = confusion_matrix(
        x["Reference Decision"],
        x["Decision"],
        labels=DECISION_LABELS
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=DECISION_LABELS
    )

    disp.plot()
    plt.title(f"{method} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(
        OUTPUT_DIR / f"{method.replace('-', '_')}_Confusion_Matrix.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 6. Operational Correctness

In [ ]:

operational_rows = []

for method in METHODS:
    x = analysis[analysis["Method"] == method].copy()

    ref_severity = x["Reference Severity"].map(SEVERITY_MAP)
    out_severity = x["Severity"].map(SEVERITY_MAP)
    severity_diff = out_severity - ref_severity

    ref_escalation = x["Reference Escalation"].map(ESCALATION_MAP)
    out_escalation = x["Escalation"].map(ESCALATION_MAP)
    escalation_diff = out_escalation - ref_escalation

    operational_rows.append({
        "Method": method,
        "Severity Accuracy": (ref_severity == out_severity).mean(),
        "Severity MAE": severity_diff.abs().mean(),
        "Severity Over": (severity_diff > 0).sum(),
        "Severity Under": (severity_diff < 0).sum(),
        "Escalation Accuracy": (ref_escalation == out_escalation).mean(),
        "Over-Escalation": (escalation_diff > 0).sum(),
        "Under-Escalation": (escalation_diff < 0).sum()
    })

operational_metrics = pd.DataFrame(operational_rows)
display(operational_metrics.round(4))

ax = operational_metrics.set_index("Method").plot(
    kind="bar",
    y=["Severity Accuracy", "Escalation Accuracy"],
    figsize=(8, 5)
)

ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("Operational Accuracy")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Operational_Accuracy.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

ax = operational_metrics.set_index("Method").plot(
    kind="bar",
    y=["Over-Escalation", "Under-Escalation"],
    figsize=(8, 5)
)

ax.set_ylabel("Scenario Count")
ax.set_title("Escalation Errors")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Escalation_Errors.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 7. Efficiency

In [ ]:

timing_rows = []

for method in METHODS:
    x = analysis[analysis["Method"] == method]
    times = pd.to_numeric(x["Time (s)"], errors="coerce").dropna()

    timing_rows.append({
        "Method": method,
        "Mean Time (s)": times.mean(),
        "Median Time (s)": times.median(),
        "Minimum Time (s)": times.min(),
        "Maximum Time (s)": times.max()
    })

timing_metrics = pd.DataFrame(timing_rows)

manual_mean = timing_metrics.loc[
    timing_metrics["Method"] == "Manual",
    "Mean Time (s)"
].iloc[0]

timing_metrics["Speed-up vs Manual"] = (
    manual_mean / timing_metrics["Mean Time (s)"]
)

display(timing_metrics.round(3))

ax = timing_metrics.plot(
    x="Method",
    y="Mean Time (s)",
    kind="bar",
    legend=False,
    figsize=(7, 5)
)

ax.set_yscale("log")
ax.set_ylabel("Mean Time (seconds, log scale)")
ax.set_title("Mean Triage / Processing Time")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Mean_Time.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

time_curve = analysis.pivot(
    index="Case ID",
    columns="Method",
    values="Time (s)"
)[METHODS]

ax = time_curve.plot(
    kind="line",
    marker="o",
    figsize=(10, 5)
)

ax.set_yscale("log")
ax.set_ylabel("Time (seconds, log scale)")
ax.set_xlabel("Scenario")
ax.set_title("Time by Scenario")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Time_By_Scenario.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

speedup = timing_metrics[
    timing_metrics["Method"] != "Manual"
][["Method", "Speed-up vs Manual"]]

ax = speedup.plot(
    x="Method",
    y="Speed-up vs Manual",
    kind="bar",
    legend=False,
    figsize=(7, 5)
)

ax.set_ylabel("Speed-up Factor (x)")
ax.set_title("Automation Speed-up vs Manual")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Automation_Speedup.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 8. LLM Reliability

In [ ]:

gemini_eval = analysis[
    analysis["Method"] == "Gemini"
].copy()

gemini_eval["Outcome Mismatch"] = (
    (gemini_eval["Decision"] != gemini_eval["Reference Decision"])
    |
    (gemini_eval["Severity"] != gemini_eval["Reference Severity"])
    |
    (gemini_eval["Escalation"] != gemini_eval["Reference Escalation"])
)

gemini_eval["High Confidence"] = (
    gemini_eval["Gemini Confidence (%)"] >= 95
)

gemini_eval["High-Confidence Error"] = (
    gemini_eval["High Confidence"]
    &
    gemini_eval["Outcome Mismatch"]
)

display(
    gemini_eval[
        [
            "Case ID",
            "Gemini Confidence (%)",
            "Outcome Mismatch",
            "High-Confidence Error"
        ]
    ]
)

high_conf = gemini_eval["High Confidence"].sum()
high_conf_errors = gemini_eval["High-Confidence Error"].sum()

llm_summary = pd.DataFrame([{
    "Gemini Cases": len(gemini_eval),
    "High Confidence Cases": high_conf,
    "High-Confidence Error Cases": high_conf_errors,
    "High-Confidence Error Rate": (
        high_conf_errors / high_conf if high_conf else np.nan
    )
}])

display(llm_summary.round(4))

ax = gemini_eval.plot(
    x="Case ID",
    y="Gemini Confidence (%)",
    kind="bar",
    legend=False,
    figsize=(10, 5)
)

ax.set_ylim(0, 105)
ax.set_ylabel("Confidence (%)")
ax.set_title("Gemini Confidence by Scenario")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Gemini_Confidence.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


## 9. Disagreement Cases

In [ ]:

disagreement_rows = []

for _, row in analysis.iterrows():

    decision_match = (
        row["Decision"] == row["Reference Decision"]
    )

    severity_match = (
        row["Severity"] == row["Reference Severity"]
    )

    escalation_match = (
        row["Escalation"] == row["Reference Escalation"]
    )

    if not (
        decision_match
        and severity_match
        and escalation_match
    ):
        disagreement_rows.append({
            "Case ID": row["Case ID"],
            "Method": row["Method"],
            "Decision Match": decision_match,
            "Severity Match": severity_match,
            "Escalation Match": escalation_match,
            "Reference Decision": row["Reference Decision"],
            "Decision": row["Decision"],
            "Reference Severity": row["Reference Severity"],
            "Severity": row["Severity"],
            "Reference Escalation": row["Reference Escalation"],
            "Escalation": row["Escalation"]
        })

disagreements = pd.DataFrame(disagreement_rows)
display(disagreements)


## 10. Export Results

In [ ]:

from google.colab import files
import shutil

RESULTS_FILE = OUTPUT_DIR / "SOC_Evaluation_Results.xlsx"
DISAGREEMENT_FILE = OUTPUT_DIR / "Qualitative_Disagreement_Cases.xlsx"

with pd.ExcelWriter(RESULTS_FILE) as writer:
    decision_metrics.to_excel(
        writer,
        sheet_name="Decision",
        index=False
    )

    operational_metrics.to_excel(
        writer,
        sheet_name="Operational",
        index=False
    )

    timing_metrics.to_excel(
        writer,
        sheet_name="Efficiency",
        index=False
    )

    llm_summary.to_excel(
        writer,
        sheet_name="LLM Reliability",
        index=False
    )

    disagreements.to_excel(
        writer,
        sheet_name="Disagreements",
        index=False
    )

disagreements.to_excel(
    DISAGREEMENT_FILE,
    index=False
)

# Copy clean dataset files into the output folder.
shutil.copy2(CLEAN_XLSX, OUTPUT_DIR / CLEAN_XLSX.name)
shutil.copy2(CLEAN_CSV, OUTPUT_DIR / CLEAN_CSV.name)

# Create one ZIP containing all result files and charts.
ZIP_BASE = DATA_DIR / "SOC_Triage_Evaluation_Output"
ZIP_FILE = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=str(OUTPUT_DIR)
)

print("Clean dataset:", CLEAN_XLSX)
print("Results:", RESULTS_FILE)
print("Disagreement cases:", DISAGREEMENT_FILE)
print("Charts folder:", OUTPUT_DIR)
print("ZIP package:", ZIP_FILE)

# Automatically trigger download to the browser.
files.download(ZIP_FILE)
